# Quran Audio Classification

In [1]:
# import kagglehub
# import os
# import shutil

# target_dir = os.path.expanduser("~/Desktop/LEARNINGPANDAS/10_Pytorch/data/Quran_recitation")

# download_path = kagglehub.dataset_download("mohammedalrajeh/quran-recitations-for-audio-classification")

# os.makedirs(os.path.dirname(target_dir), exist_ok=True)

# if os.path.exists(target_dir):
#     shutil.rmtree(target_dir)

# shutil.move(download_path, target_dir)

# print(f"Dataset successfully moved to: {target_dir}")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import librosa
from sklearn.preprocessing import LabelEncoder
import os
import time
from skimage.transform import resize 

In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [4]:
df = pd.read_csv('/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/Quran_recitation/files_paths.csv')
df.head()

,FilePath,Class
0,./Dataset/Mohammed_Aluhaidan/lohaidan_171.wav,Mohammed_Aluhaidan
1,./Dataset/Mohammed_Aluhaidan/lohaidan_159.wav,Mohammed_Aluhaidan
2,./Dataset/Mohammed_Aluhaidan/lohaidan_401.wav,Mohammed_Aluhaidan
3,./Dataset/Mohammed_Aluhaidan/lohaidan_367.wav,Mohammed_Aluhaidan
4,./Dataset/Mohammed_Aluhaidan/lohaidan_373.wav,Mohammed_Aluhaidan


In [5]:
df['Class'].unique()

array(['Mohammed_Aluhaidan', 'Yasser_Aldossary', 'Maher_Almuaiqly',
       'Nasser_Alqutami', 'AbdulBari_Althubaity', 'Bander_Balilah',
       'Ali_Alhothaify', 'Saud_Alshuraim', 'Mohammed_Ayoub',
       'AbdulRahman_Alsudais', 'Saad_Alghamdi', 'Abdullah_Albuaijan'],
      dtype=object)

In [6]:
df['FilePath'] = '/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/Quran_recitation/Dataset' + df['FilePath'].str[1:]
df

,FilePath,Class
0,/Users/atharvakhandelwal/Desktop/learningpanda...,Mohammed_Aluhaidan
1,/Users/atharvakhandelwal/Desktop/learningpanda...,Mohammed_Aluhaidan
2,/Users/atharvakhandelwal/Desktop/learningpanda...,Mohammed_Aluhaidan
3,/Users/atharvakhandelwal/Desktop/learningpanda...,Mohammed_Aluhaidan
4,/Users/atharvakhandelwal/Desktop/learningpanda...,Mohammed_Aluhaidan
...,...,...
6682,/Users/atharvakhandelwal/Desktop/learningpanda...,Abdullah_Albuaijan
6683,/Users/atharvakhandelwal/Desktop/learningpanda...,Abdullah_Albuaijan
6684,/Users/atharvakhandelwal/Desktop/learningpanda...,Abdullah_Albuaijan
6685,/Users/atharvakhandelwal/Desktop/learningpanda...,Abdullah_Albuaijan


In [7]:
df.iloc[0,0]

'/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/Quran_recitation/Dataset/Dataset/Mohammed_Aluhaidan/lohaidan_171.wav'

In [8]:
l_encoder = LabelEncoder()
df['Class'] = l_encoder.fit_transform(df['Class'])

In [9]:
train = df.sample(frac=0.7)
test = df.drop(train.index)
val = test.sample(frac=0.5)
test = test.drop(val.index)

In [10]:
print('Train:',train.shape)
print('Test:',test.shape)
print('Val:',val.shape)

Train: (4681, 2)
Test: (1003, 2)
Val: (1003, 2)


In [11]:
class CustomDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.label = torch.tensor(dataframe['Class'].values).type(torch.LongTensor)

    def __len__(self):
        return self.dataframe.shape[0]

    def __getitem__(self, index):
        img_path = self.dataframe.iloc[index, 0]
        label = self.label[index]
        spec = self.get_spectogram(img_path)
        audio = torch.tensor(spec).type(torch.FloatTensor).unsqueeze(0)
        
        return audio, label
    
    def get_spectogram(self, file_path):
        sr = 22050
        duration = 5
        img_h, img_w = 128, 256

        try:
            signal, sr = librosa.load(file_path, sr=sr, duration=duration)
            
            expected_len = sr * duration
            if len(signal) < expected_len:
                signal = librosa.util.fix_length(signal, size=expected_len)
                
            spec = librosa.feature.melspectrogram(y=signal, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
            spec_db = librosa.power_to_db(spec, ref=np.max)
            
            spec_resized = resize(spec_db, (img_h, img_w), anti_aliasing=True)
            return spec_resized
        except Exception as e:
            print(f"Error: {e}")
            return np.zeros((img_h, img_w))

In [12]:
train_dataset= CustomDataset(dataframe=train)
test_dataset= CustomDataset(dataframe=test)
val_dataset= CustomDataset(dataframe=val)

In [13]:
LR = 1e-4
batch_size = 32
epochs = 25

In [14]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [15]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pooling = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()

        self.linear1 = nn.Linear(64 * 16 * 32, 4096)
        self.linear2 = nn.Linear(4096, 1024)
        self.linear3 = nn.Linear(1024, 512)
        self.output = nn.Linear(512, 12)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pooling(x)
        
        x = self.conv2(x)
        x = self.relu(x) 
        x = self.pooling(x)
        
        x = self.conv3(x)
        x = self.relu(x) 
        x = self.pooling(x)
        
        x = self.flatten(x)
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.linear2(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.linear3(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.output(x)

        return x

In [16]:
model = Net().to(device)

In [17]:
model

Net(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pooling): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=32768, out_features=4096, bias=True)
  (linear2): Linear(in_features=4096, out_features=1024, bias=True)
  (linear3): Linear(in_features=1024, out_features=512, bias=True)
  (output): Linear(in_features=512, out_features=12, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [19]:
from tqdm import tqdm
import torch

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
    
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
        pbar.set_postfix(loss=loss.item(), acc=100 * correct / total)

print("Training Complete!")

Epoch 25/25: 100%|██████████| 147/147 [01:27<00:00,  1.69batch/s, acc=99.7, loss=0.00185] 

Training Complete!


In [21]:
# 2. Validation Loop
model.eval()
val_loss = 0.0
val_correct = 0
val_total = 0

print("Starting Validation...")

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        val_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

print(f"Validation Results - Loss: {val_loss/len(val_loader):.4f}, Accuracy: {100 * val_correct / val_total:.2f}%")

Starting Validation...
Validation Results - Loss: 0.0297, Accuracy: 99.00%


In [23]:
# 3. Test Loop
model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0

print("Starting Testing...")

with torch.no_grad():
    
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"Test Set Results - Loss: {test_loss/len(test_loader):.4f}, Accuracy: {100 * test_correct / test_total:.2f}%")

Starting Testing...
Test Set Results - Loss: 0.0491, Accuracy: 98.50%
